In [1]:
import sympy as sym
import numpy as np
from functools import reduce

In [2]:
s, b0, a0, a1, a2, a3, a4 = sym.symbols("s b_0 a_0 a_1 a_2 a_3 a_4")

In [3]:
G = b0 / (s**5 + a4 * s**4 + a3 * s**3 + a2 * s**2 + a1 * s + a0)
G

b_0/(a_0 + a_1*s + a_2*s**2 + a_3*s**3 + a_4*s**4 + s**5)

In [4]:
z, T = sym.symbols("z T")

In [5]:
euler = (z - 1) / T
implicito = (z - 1) / (z * T)
tustin = (2 / T) * (z - 1) / (z + 1)

tustin

2*(z - 1)/(T*(z + 1))

In [6]:
def analisis(metodo):
    # coef_dem = sym.poly(dem, z).all_coeffs()
    Gd = G.subs([ (s, metodo) ])
    num, dem = sym.fraction(Gd.simplify())

    k = sym.symbols("k")
    u = sym.Function("u", integer = True)(k)
    p = sym.Function("p_c", integer = True)(k)
    funciones_posicion = []

    coef_num = sym.poly(num, z).all_coeffs()
    for i, coef in enumerate(coef_num[::-1]):
        coef_num[i] = u.subs([ (k, k - i) ]) * coef.subs([ (b0, 1), (T, 1) ])
    funcion_control = reduce(lambda x, y: x + y, coef_num)
    valores_control = b0 * T**5 * funcion_control

    coef_dem = sym.poly(dem, z).all_coeffs()

    constantes = [a0, a1, a2, a3, a4]

    matrix = []
    for coef in coef_dem:
        independiente = coef.subs([ (a, 0) for a in constantes ])
        coef = coef - independiente

        fila = []
        for i in range(len(constantes)):
            coef_actual = coef.subs([ (a, 1) if i == j else (a, 0) for j, a in enumerate(constantes) ])
            fila.append(coef_actual)

        fila.append(independiente)
        matrix.append(fila)

    trans_c2d = sym.Matrix(matrix)
    trans_d2c = trans_c2d.inv()

    alfa0, alfa1, alfa2, alfa3, alfa4, alfa5 = sym.symbols("\\alpha_0 \\alpha_1 \\alpha_2 \\alpha_3 \\alpha_4 \\alpha_5")
    alfas = sym.Matrix([ alfa0, alfa1, alfa2, alfa3, alfa4, alfa5 ])

    # Existe una relacion entre los alfas, la vamos a desarmar, dado por el a_vec[5] = 1
    # alfa_5 = f(alfa_1, alfa_2, alfa_3, alfa_4)
    relacion_alfa5 = sym.solve((trans_d2c * alfas)[5].simplify() - 1, alfa5)[0]
    alfa5 = relacion_alfa5
    alfas[-1] = alfa5

    posiciones = []
    for i, alfa in enumerate(alfas):
        funcion = p.subs([ (k, k - i) ])
        posiciones.append(funcion * alfa)
        funciones_posicion.append(funcion)
    valores_posicion = reduce(lambda x, y: x + y, posiciones)

    coef_posicion = []
    coef_constante = valores_posicion.subs([ (alfa, 0) for alfa in alfas[:-1] ])
    for i in range(len(alfas) - 1):
        sustitucion = [ (alfa, alfa if i == j else 0) for j, alfa in enumerate(alfas[:-1]) ]
        coef_posicion.append((valores_posicion - coef_constante).subs(sustitucion).simplify())

    lado_izquierdo = coef_posicion[0]
    coef_posicion[0] = coef_constante

    relacion_coeficientes = {
        "A": sym.Matrix(trans_d2c[:-1, :]),
    }

    for i, coef in enumerate(coef_posicion):
        alfa = alfas[i]
        peso, funcion = sym.factor_terms(coef).as_coeff_Mul()
        expresion = peso * alfa
        if i == 0: # Es el coeficiente 
            expresion = peso

        relacion_coeficientes[f"D_{i + 1}"] = (alfa, expresion / alfa0, funcion.subs([ (alfa, 1) ]))

    relacion_coeficientes[f"D_{len(coef_posicion) + 1}"] = (b0, valores_control.subs([ (funcion_control, 1) ]) / alfa0, funcion_control)

    lado_derecho = valores_control - (reduce(lambda x, y: x + y, coef_posicion))

    return sym.Eq(lado_izquierdo, lado_derecho.simplify()), relacion_coeficientes

ecuacion_diferencias, relacion_coeficientes = analisis(implicito)

# Resultado

In [7]:
render = print
new_print = print
try: # Por si no esta instalado, de alguna forma, debería estar si usar jupyter notebook
    from IPython.display import display, Math, Latex
    render = lambda arg: display(Math(arg))
    new_print = lambda arg: display(Latex(arg))
except:
    print("Usamos default render aka print")

In [8]:
ecuacion_diferencias

Eq(\alpha_0*p_c(k), T**5*b_0*u(k - 5) - \alpha_1*p_c(k - 1) - \alpha_2*p_c(k - 2) - \alpha_3*p_c(k - 3) - \alpha_4*p_c(k - 4) + p_c(k - 5))

In [9]:
for coef_regresion, par_exp_funcion in relacion_coeficientes.items():
    if coef_regresion == "D" or coef_regresion == "A":
        continue

    variable, exp, funcion = par_exp_funcion
    ecuacion = sym.Eq(sym.symbols(coef_regresion), exp)
    despeje = sym.solveset(ecuacion, variable)

    render(f"{sym.latex(ecuacion)} \\cdot \\left({sym.latex(funcion)}\\right) \\implies {sym.latex(variable)} = {sym.latex(despeje)}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [10]:
a_vec = sym.Matrix([ sym.symbols(f"a_{i}") for i in range(5) ])
alfa_vec = sym.Matrix([ sym.symbols(f"\\alpha_{i}") for i in range(5) ])
render(f"{sym.latex(a_vec)} = {sym.latex(relacion_coeficientes["A"])} {sym.latex(alfa_vec)}")

<IPython.core.display.Math object>